In [2]:
import pandas as pd

In [30]:
# Read all kpler data excel files from a folder and concatenate them into a single dataframe, there are dozens of files, so we need to read them in a loop and concatenate them.
import os

folder_path = '../data/kpler_trade_flow_data/'
all_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.xlsx')]

dfs = []
for f in all_files:
    try:
        dfs.append(pd.read_excel(f, engine='openpyxl'))
    except Exception as e:
        print(f"Skipped {f}: {e}")

kpler_data = pd.concat(dfs, ignore_index=True)
kpler_data.shape

Skipped ../data/kpler_trade_flow_data/~$export_2025.03.24.xlsx: File is not a zip file


(22262, 178)

In [55]:
kpler_data.to_excel('../data/Processed_data/kpler_trade_flow_data_combined.xlsx', index=False)

In [ ]:
# Count how many unique sellers-buyers pairs, we have sellers and buyers in separate columns, we want to count the unique pairs of sellers and buyers.
kpler_data['seller_buyer_pair'] = kpler_data['Seller (origin)'] + ' - ' + kpler_data['Buyer (destination)']
unique_pairs = kpler_data['seller_buyer_pair'].nunique()
print(f'The number of unique seller-buyer pairs is: {unique_pairs}')    
print(kpler_data['Seller (origin)'].nunique())
print(kpler_data['Buyer (destination)'].nunique())


# Output the unique sellers-port pairs
kpler_data['seller_port_pair'] = kpler_data['Seller (origin)'] + ' - ' + kpler_data['Installation origin']
unique_seller_port_pairs = kpler_data['seller_port_pair'].nunique()
print(f'The number of unique seller-port pairs is: {unique_seller_port_pairs}')

# Get kpler dataframe without blank values in Seller (origin), Buyer (Destination), Installation origin, Installation Destination, and Product, and count the unique seller-port pairs again
kpler_data_no_unknown = kpler_data.dropna(subset=['Seller (origin)', 'Buyer (destination)', 'Installation origin', 'Installation Destination', 'Product'])
print(kpler_data_no_unknown.shape)

# print unique seller-installation origin-buyer-installation destination-product pairs, for those with blank, we will fill them with "Unknown"
kpler_data_no_unknown['seller_buyer_product_pair'] = kpler_data_no_unknown['Seller (origin)'] + ' - ' + \
    kpler_data_no_unknown['Installation origin'] + ' - ' + kpler_data_no_unknown['Buyer (destination)']\
        + ' - ' + kpler_data_no_unknown['Installation Destination'] + ' - ' + kpler_data_no_unknown['Product']
unique_seller_buyer_product_pairs = kpler_data_no_unknown['seller_buyer_product_pair'].nunique()
print(f'The number of unique seller-installation origin-buyer-installation destination-product pairs is: {unique_seller_buyer_product_pairs}')

The number of unique seller-buyer pairs is: 1018
208
318
The number of unique seller-port pairs is: 240
(6815, 180)
The number of unique seller-installation origin-buyer-installation destination-product pairs is: 1992


/var/folders/rh/5tkty3715h119sm4y35zw4dr0000gn/T/ipykernel_4663/1927661393.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  kpler_data_no_unknown['seller_buyer_product_pair'] = kpler_data_no_unknown['Seller (origin)'] + ' - ' + \


In [37]:
kpler_data_no_unknown.head()

,Vessel,Country (origin),Zone Origin,Installation origin,End (origin),Country (destination),Zone Destination,Installation Destination,Start (destination),Cargo (tons),...,Forecasted origins confidence,Forecasted destinations,Forecasted destinations confidence,Bill Number,Shipper,Consignee,Notify Party,seller_buyer_pair,seller_port_pair,seller_buyer_product_pair
5,Herbert C Jackson,United States,Marquette,LS&I Ore Dock,2025-04-06 22:05,United States,Detroit,Cleveland Cliffs Dearborn Works,2025-04-09 09:42,5950,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cleveland-Cliffs - AK Steel,Cleveland-Cliffs - LS&I Ore Dock,Cleveland-Cliffs - LS&I Ore Dock - AK Steel - ...
9,Feng Hua,Australia,Port Hedland,Finucane Island,2025-04-06 20:35,China,Luojing,Taicang Ore,2025-04-25 04:53,86749,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BHP - Shagang Group,BHP - Finucane Island,BHP - Finucane Island - Shagang Group - Taican...
10,Feng Hua,Australia,Port Hedland,Finucane Island,2025-04-06 20:35,China,Rizhao Port,Rizhao Dry Bulks,2025-04-21 03:49,86749,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BHP - Shagang Group,BHP - Finucane Island,BHP - Finucane Island - Shagang Group - Rizhao...
17,Amnsi Stallion,India,Paradip,Paradip Central Quay Terminal,2025-04-06 18:13,India,Hazira,AM/NS Hazira,2025-04-16 12:14,78712,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AM/NS - AM/NS,AM/NS - Paradip Central Quay Terminal,AM/NS - Paradip Central Quay Terminal - AM/NS ...
21,Star Tembu,Australia,Cape Preston,Sino Iron,2025-04-06 12:53,China,Ningbo,Ningbo Steel Plant,2025-04-23 06:35,55545,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Citic Pacific Mining - Baosteel,Citic Pacific Mining - Sino Iron,Citic Pacific Mining - Sino Iron - Baosteel - ...


In [ ]:
# df = pd.read_excel('../data/Processed_data/Iron_mines_data_w_cost_v2.xlsx', index_col=False)
# # convert the 'Capacity_2025' column to numeric, and fill the blank and unknown values with 0
# df['Capacity_2025'] = pd.to_numeric(df['Capacity_2025'], errors='coerce').fillna(0)
# # Calcualte the total capacity of the mines with non-blank Fe Content (%)
# capacity_SP = df[df['Fe Content (%)'].notna()]['Capacity_2025'].sum()
# # Calcualte the percentage of capacity_SP over total capacity of all mines in the global tracker data
# capacity_total = df['Capacity_2025'].sum()
# percentage = capacity_SP / capacity_total * 100
# print(f'The total capacity of the mines with non-blank Fe Content (%) is: {capacity_SP} million tons, which accounts for {percentage:.2f}% of the total capacity of')

The total capacity of the mines with non-blank Fe Content (%) is: 3536196.0 million tons, which accounts for 54.82% of the total capacity of


In [39]:
# Aggregate trade flow by seller-buye-product pairs, and sum the Cargo (tons), output the entire agregated dataframe to a new excel file, need to keep other columns as well
agg_dict = {col: 'first' for col in kpler_data.columns if col != 'seller_buyer_product_pair'}
agg_dict['Cargo (tons)'] = 'sum'
# agg_df should keep the column of seller_buyer_product_pair
agg_df = kpler_data_no_unknown.groupby('seller_buyer_product_pair').agg(agg_dict).reset_index()
agg_df.to_excel('../data/Processed_data/kpler_aggregated_trade_flow.xlsx', index=False)

In [51]:
# Aggregate trade flow by seller-buye-product pairs, and sum the Cargo (tons), output the entire agregated dataframe to a new excel file, need to keep other columns as well
agg_dict = {col: 'first' for col in kpler_data_no_unknown.columns if col != 'seller_buyer_product_pair'}
agg_dict['Cargo (tons)'] = 'sum'
# agg_df should keep the column of seller_buyer_product_pair
agg_df = kpler_data_no_unknown.groupby('seller_buyer_product_pair').agg(agg_dict).reset_index()

# For agg_df, separate the sellers inthe Seller (origin) column by comma and explode the dataframe so that each row only has one seller, and do the same for buyers in the Buyer (destination) column, and keep other columns the same. Noted that we need to keep all sellers rather than just the first seller. For the cargo (tons) column, we need to average the original cargo (tons) for the same seller-buyer-product pair, rather than just taking the first value, because after we explode the sellers and buyers, we will have multiple rows for the same seller-buyer-product pair, and we want to keep the total cargo (tons) for that pair. After exploding, output the entire dataframe to a new excel file.
# Count combinations before exploding to split cargo correctly
agg_df['_n_sellers'] = agg_df['Seller (origin)'].str.split(',').str.len().fillna(1)
agg_df['_n_buyers']  = agg_df['Buyer (destination)'].str.split(',').str.len().fillna(1)
agg_df['_n_combos']  = agg_df['_n_sellers'] * agg_df['_n_buyers']

# Split into lists
agg_df_exploded = agg_df.assign(
    **{'Seller (origin)':    agg_df['Seller (origin)'].str.split(','),
       'Buyer (destination)': agg_df['Buyer (destination)'].str.split(',')}
)

# Explode separately → cross product (each seller paired with each buyer)
agg_df_exploded = agg_df_exploded.explode('Seller (origin)', ignore_index=True)
agg_df_exploded = agg_df_exploded.explode('Buyer (destination)', ignore_index=True)

# Divide cargo by number of seller-buyer combos
agg_df_exploded['Cargo (tons)'] = agg_df_exploded['Cargo (tons)'] / agg_df_exploded['_n_combos']

# Cleanup
agg_df_exploded['Seller (origin)']    = agg_df_exploded['Seller (origin)'].str.strip()
agg_df_exploded['Buyer (destination)'] = agg_df_exploded['Buyer (destination)'].str.strip()
agg_df_exploded = agg_df_exploded.drop(columns=['_n_sellers', '_n_buyers', '_n_combos'])

agg_df_exploded.to_excel('../data/Processed_data/kpler_aggregated_trade_flow_exploded.xlsx', index=False)


In [56]:
# Reasoning the iron ore mine plant based on the columns including 'Seller (origin)', 'Installation origin', 'Zone Origin' and 'Country (origin)'
extracted_seller_data = agg_df_exploded[['Seller (origin)', 'Installation origin', 'Zone Origin', 'Country (origin)']]
extracted_seller_data.to_excel('../data/Processed_data/kpler_extracted_seller_data.xlsx', index=False)
# extracted_buyer_data = agg_df_exploded[['Buyer (destination)', 'Installation Destination', 'Zone Destination', 'Country (destination)']]
# extracted_buyer_data.to_excel('../data/Processed_data/kpler_extracted_buyer_data.xlsx', index=False)


In [52]:
agg_df_exploded.shape

(2418, 181)